<a href="https://colab.research.google.com/github/tripti369/Titech-_TriptiTiwari/blob/main/Titech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install gradio
!pip install unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wb0m8sb3/unsloth_e196dd3aacd244769269adb7182e7509
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wb0m8sb3/unsloth_e196dd3aacd244769269adb7182e7509
  Resolved https://github.com/unslothai/unsloth.git to commit 5fa8683b27333111657e1976166fabce7e4abed5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.4-py3-none-any.whl size=29553786 sha256=42147aae45a3eb63cd99f30b06151444ca2902b1abb05ecf8a4852dad0364f84
  Stored in directory: /tmp/pip-ephem-wheel-cache-__f7egzg/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━

In [2]:
!unzip -o '/content/archive (1).zip'

unzip:  cannot find or open /content/archive (1).zip, /content/archive (1).zip.zip or /content/archive (1).zip.ZIP.


In [6]:
import zipfile
import os
import json
from datasets import Dataset

# 1. Extract the Zip
zip_path = "/content/archive.zip" # <-- RENAME THIS TO YOUR ACTUAL ZIP FILE NAME
extract_path = "dataset_files"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 2. Process Files
# This assumes your data is in a JSON or TXT format.
# If it's just raw text, we'll use the curated data below as a fallback + your data.
processed_data = []

# Fallback/Seed Data to ensure high quality
seed_data = [
    {"input": "Lessee shall indemnify Lessor...", "output": "You pay for damages you cause."},
    {"input": "Joint and several liability...", "output": "You are responsible if your roommate doesn't pay."},
]

# Logic to read your files (Adjust based on your actual file content)
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".json"):
            with open(os.path.join(root, file), 'r') as f:
                content = json.load(f)
                # Assuming your JSON has 'clause' and 'explanation' keys
                if isinstance(content, list):
                    processed_data.extend(content)

# Combine and Format
final_list = processed_data if processed_data else seed_data

def format_prompt(sample):
    # Standardizing for Gemma-3
    return {"text": f"### Instruction:\nSimplify this rental clause\n\n### Input:\n{sample.get('input', sample.get('clause'))}\n\n### Response:\n{sample.get('output', sample.get('explanation'))}"}

dataset = Dataset.from_list(final_list).map(format_prompt)
print(f"Dataset ready with {len(dataset)} examples!")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset ready with 2 examples!


In [7]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
import gradio as gr

# 1. Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

# 2. Train
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 40,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)
trainer.train()

# 3. Launch UI for Demo
def predict(clause):
    prompt = f"### Instruction:\nSimplify this rental clause\n\n### Input:\n{clause}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip()

gr.Interface(fn=predict, inputs="text", outputs="text", title="LegalEase 270M").launch(share=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 40 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,474,560 of 269,572,736 (0.55% trained)


Step,Training Loss
1,5.702069
2,5.187670
3,4.729927
4,4.345291
5,4.018859
6,3.722873
7,3.442068
8,3.171547
9,2.912485
10,2.659154


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://519a3dae810580dc87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
import gradio as gr

# 1. Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

# 2. Train
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 40,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)
trainer.train()

# 3. Launch UI for Demo
def predict(clause):
    prompt = f"### Instruction:\nSimplify this rental clause\n\n### Input:\n{clause}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip()

gr.Interface(fn=predict, inputs="text", outputs="text", title="LegalEase 270M").launch(share=True)

==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 40 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,474,560 of 269,572,736 (0.55% trained)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12

Step,Training Loss
1,5.702069
2,5.185566
3,4.734728
4,4.351258
5,4.028909
6,3.730731
7,3.452397
8,3.182671
9,2.921551
10,2.671271


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91fc06741e98b8b209.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
